# Pix2Pix Inference — v3 Test Split

Runs inference on the held-out test set for both models:
- **HE_PAS_model_v3** : HE → PAS
- **HE_TRI_model_v3** : HE → TRI

Then displays a grid of Real A | Fake B | Real B for visual evaluation.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
REPO      = r'' # path to pix2pix model
DATA_ROOT = r'' # path to data
CHECKPOINTS_DIR = r'' # path to checkpoints

DIRECTION = 'AtoB'
GPU_ID    = 0
NUM_TEST  = 10000   # large number to process all test patches

# How many patches to show in the visual grid
N_SHOW = 8

In [ ]:
%cd "" # path to change directory to

## 1. Run Inference — HE → PAS

In [ ]:
!python "{REPO}/test.py" \
    --dataroot "{DATA_ROOT}/HE_PAS/AB" \
    --name HE_PAS_model_v3 \
    --model pix2pix \
    --direction {DIRECTION} \
    --dataset_mode aligned \
    --checkpoints_dir "{CHECKPOINTS_DIR}" \
    --phase test \
    --num_test {NUM_TEST} 

## 2. Run Inference — HE → TRI

In [ ]:
!python "{REPO}/test.py" \
    --dataroot "{DATA_ROOT}/HE_TRI/AB" \
    --name HE_TRI_model_v3 \
    --model pix2pix \
    --direction {DIRECTION} \
    --dataset_mode aligned \
    --checkpoints_dir "{CHECKPOINTS_DIR}" \
    --phase test \
    --num_test {NUM_TEST}

## 3. Visualize Results

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


def visualize_results(model_name, task_label, n_show=N_SHOW):
    """
    Load results from the pix2pix results folder and display
    a grid of:  Real A (HE)  |  Fake B (generated)  |  Real B (target)
    """
    results_dir = os.path.join(
        REPO, 'results', model_name, 'test_latest', 'images'
    )

    if not os.path.exists(results_dir):
        print(f"Results not found at: {results_dir}")
        print("Make sure inference ran successfully.")
        return

    fake_files = sorted([f for f in os.listdir(results_dir) if 'fake_B' in f])
    n_total    = len(fake_files)
    print(f"{model_name}: {n_total} test patches found")

    # Sample evenly across the test set
    indices    = np.linspace(0, n_total - 1, n_show, dtype=int)
    show_files = [fake_files[i] for i in indices]

    fig, axes = plt.subplots(n_show, 3, figsize=(13, n_show * 4.5))
    fig.suptitle(
        f'{task_label} — {model_name}\n'
        f'Showing {n_show} of {n_total} test patches (sampled evenly)',
        fontsize=12, y=1.01
    )

    col_titles = ['Real A (HE input)', 'Fake B (generated)', 'Real B (ground truth)']

    for row, fname in enumerate(show_files):
        base   = fname.replace('_fake_B.png', '')
        real_a = cv2.cvtColor(
            cv2.imread(os.path.join(results_dir, base + '_real_A.png')),
            cv2.COLOR_BGR2RGB
        )
        fake_b = cv2.cvtColor(
            cv2.imread(os.path.join(results_dir, fname)),
            cv2.COLOR_BGR2RGB
        )
        real_b = cv2.cvtColor(
            cv2.imread(os.path.join(results_dir, base + '_real_B.png')),
            cv2.COLOR_BGR2RGB
        )

        for col, (title, img) in enumerate(zip(col_titles,
                                               [real_a, fake_b, real_b])):
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(title, fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()


visualize_results('HE_PAS_model_v3', 'HE → PAS')
visualize_results('HE_TRI_model_v3', 'HE → TRI')

## 4. Save Results to Output Folder

In [ ]:
import shutil

RESULTS_OUT = r'' # path to save results to

for model_name in ('HE_PAS_model_v3', 'HE_TRI_model_v3'):
    src = os.path.join(REPO, 'results', model_name)
    dst = os.path.join(RESULTS_OUT, model_name)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Copied {model_name} -> {dst}")
    else:
        print(f"Not found: {src}")